# 06 · Equity analysis

Who has a 5-minute walk (≤ 400 m) to each service, and who doesn't, by income, car ownership and age.
Same code as `make equity` (`urban_access.equity.run`).

**Method.** Access is measured per home (Phase 2 network distances, population allocated to homes).
Groups come from ACS 5-year block-group **counts**: all residents (`B01003`), households without a
vehicle (`B25044`), residents 65+ (`B01001`) and residents below 200 % of the poverty line (`C17002`,
ratio of income to poverty level; "low income" below). A group's share within a band is each block
group's resident share within that band, weighted by the block group's count of the group. This
assumes the group's members are spread over the block group's homes like all residents (ecological
inference: block groups, not people, are observed).

**Uncertainty.** Block-group ACS estimates have large margins of error. Each count and the median
income are drawn 1,000 times from Normal(estimate, MOE / 1.645), clipped at 0, and every number is
recomputed per draw. Ranges are the 5th–95th percentiles of the draws. The draws add noise to values
that already contain sampling error, so they mix block groups across income terciles and weaken
correlations: the range shows how robust a result is to ACS error, and the published-value point
estimate can sit at its edge. Draws are independent (no covariance between numerator and denominator).

In [ ]:
import os

import urban_access  # noqa: F401  (first: sets PROJ_DATA before geopandas/pyogrio load)
from urban_access.config import load_city

CITY = os.environ.get("CITY", "carson_city")
cfg = load_city(CITY)
print(cfg.name, "| Overture", cfg.overture_release, "| ACS", cfg.acs_year)
print("schema:", cfg.slug, "| EPSG", cfg.analysis_epsg)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from sqlalchemy import text

from urban_access import db, equity

eng = db.engine()


def q(sql: str) -> pd.DataFrame:
    with eng.connect() as con:
        return pd.read_sql(text(sql.replace("__S__", f'"{cfg.slug}"')), con)


r = equity.run(cfg)
meta, groups = r["meta"], r["groups"]
near, far = meta["near_m"], meta["far_m"]
svc = cfg.scoring.scored_services
with eng.connect() as con:
    bg = gpd.read_postgis(text(f'SELECT * FROM "{cfg.slug}".bg_scores'), con, geom_col="geom")
bg = bg.merge(r["per_bg"], on="geoid", how="left")
print({k: meta[k] for k in ("block_groups", "residents", "mc_draws", "seed", "poverty_ratio")})

## Sanity checks

In [ ]:
# all-resident shares equal the city-wide figures computed directly from homes (notebook 05)
od = q("""SELECT od.service, od.dist_m, o.pop FROM __S__.origin_dist od
          JOIN __S__.origins o ON o.id = od.origin_id WHERE o.kind <> 'centroid'""")
direct = (
    od.assign(n=od["dist_m"] <= near)
    .groupby("service")
    .apply(lambda x: 100 * np.average(x["n"], weights=x["pop"]), include_groups=False)
)
allr = groups[(groups.group == "all_residents") & (groups.kind == "pct")].set_index("metric")
for s, v in direct.items():
    assert abs(allr.loc[f"{s}_le_{near:g}m", "point"] - v) < 1e-6, s
# group totals equal the ACS sums over the scored, populated block groups
pop = bg[bg["access_score"].notna()]
for g, col in equity.GROUPS.items():
    assert groups.loc[groups.group == g, "group_total"].iloc[0] == pop[col].sum()
print("ok:", direct.round(1).to_dict())

## Headline: residents within a 5-minute walk

In [ ]:
h = pd.DataFrame(equity.headline(groups, near, far))
h["residents"] = h["residents"].round(-1).astype(int)
h[["pct", "pct_p05", "pct_p95"]] = h[["pct", "pct_p05", "pct_p95"]].round(1)
h[["metric", "residents", "pct", "pct_p05", "pct_p95"]]

In [ ]:
row = h.set_index("metric")
g4 = row.loc[f"grocery_le_{near:g}m"]
print(
    f"{int(g4.residents):,} residents ({g4.pct:.1f} %, range {g4.pct_p05:.1f}–{g4.pct_p95:.1f}) "
    f"live within {near:g} m of a full-service grocery store; "
    f"{100 - g4.pct:.1f} % do not."
)
g0 = row.loc[f"grocery_none_le_{far:g}m"]
print(f"{int(g0.residents):,} residents ({g0.pct:.1f} %) have no grocery store within {far:g} m.")
c4 = row.loc[f"convenience_not_grocery_le_{near:g}m"]
print(
    f"{int(c4.residents):,} residents ({c4.pct:.1f} %) have a convenience store but no grocery "
    f"within {near:g} m."
)

## By group: share within 400 m, per service

In [ ]:
LABELS = {
    "all_residents": "All residents",
    "low_income_residents": "Below 200 % poverty",
    "zero_car_households": "Zero-car households",
    "residents_65plus": "Residents 65+",
}
pct = groups[(groups.kind == "pct") & groups.metric.isin([f"{s}_le_{near:g}m" for s in svc])].copy()
pct["service"] = pct["metric"].str.replace(f"_le_{near:g}m", "", regex=False)
pct["cell"] = pct.apply(lambda x: f"{x.point:.1f} [{x.p05:.1f}–{x.p95:.1f}]", axis=1)
tab = pct.pivot(index="service", columns="group", values="cell")[list(LABELS)]
tab.columns = [LABELS[c] for c in tab.columns]
tab

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(svc))
wid = 0.2
colors = ["#4d4d4d", "#1b9e77", "#d95f02", "#7570b3"]
for i, (g, lab) in enumerate(LABELS.items()):
    d = pct[pct.group == g].set_index("service").loc[svc]
    ax.bar(
        x + (i - 1.5) * wid,
        d.point,
        wid,
        label=lab,
        color=colors[i],
        yerr=[d.point - d.p05, d.p95 - d.point],
        capsize=2,
        error_kw={"lw": 0.8},
    )
ax.set_xticks(x, svc)
ax.set_ylabel(f"% within {near:g} m (walk)")
ax.set_title(f"Share within a 5-minute walk, by group (bars: MC 5–95 %, {meta['mc_draws']} draws)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

In [ ]:
# composite score by group (group-weighted mean of block-group access_score) and group sizes
m = groups[(groups.kind == "mean_score") & (groups.metric == "access_score")].set_index("group")
m = m.loc[
    list(LABELS), ["point", "p05", "p95", "group_total", "group_total_p05", "group_total_p95"]
]
m.index = [LABELS[g] for g in m.index]
m.round(1)

## Income terciles (thirds of residents by block-group median household income)

In [ ]:
t = r["terciles"]
cols = [
    "tercile",
    "income_min",
    "income_max",
    "n_block_groups",
    "residents",
    "access_score",
    "access_score_p05",
    "access_score_p95",
]
for s in svc:
    cols += [f"{s}_le_{near:g}m_pct"]
t[cols].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
lab = [
    f"T{k}\n${a / 1000:.0f}k–{b / 1000:.0f}k"
    for k, a, b in zip(t.tercile, t.income_min, t.income_max, strict=True)
]
ax.bar(
    lab,
    t.access_score,
    color="#3182bd",
    yerr=[
        (t.access_score - t.access_score_p05).clip(lower=0),
        (t.access_score_p95 - t.access_score).clip(lower=0),
    ],
    capsize=3,
)
ax.set_ylabel("mean access_score (resident-weighted)")
ax.set_title("Access by income tercile (block-group median household income)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## Correlations across block groups (Spearman, n = 45)

In [ ]:
c = r["correlations"]
c = c[c.score.isin(["access_score", "grocery_score", "transit_score", "healthcare_score"])]
c.pivot(index="variable", columns="score", values="rho").round(2)

In [ ]:
c[["variable", "score", "rho", "p_value", "rho_p05", "rho_p50", "rho_p95"]].round(3)

In [ ]:
xs = {
    "median_hh_income": "median household income ($)",
    "below_pov_pct": "% below 200 % of poverty line",
    "zero_car_hh_pct": "% households without a vehicle",
    "pop_65plus_pct": "% residents 65+",
}
fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)
for ax, (col, lab) in zip(axes, xs.items(), strict=True):
    p = bg[bg.access_score.notna()]
    ax.scatter(p[col], p.access_score, s=p.pop_total / 30, alpha=0.6, color="#3182bd")
    rho = r["correlations"].query("variable == @col and score == 'access_score'").rho.iloc[0]
    ax.set_title(f"ρ = {rho:.2f}")
    ax.set_xlabel(lab)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("access_score")
plt.tight_layout()

## Spatial clusters (Moran's I, LISA; Queen contiguity, row-standardised)

In [ ]:
pd.DataFrame(r["moran"])[
    ["variable", "n", "moran_i", "expected_i", "p_sim", "clusters", "clusters_fdr"]
]

In [ ]:
CL = {"HH": "#d7191c", "LL": "#2c7bb6", "HL": "#fdae61", "LH": "#abd9e9", "ns": "#eeeeee"}
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, (col, title) in zip(
    axes, [("lisa_cluster", "access_score"), ("grocery_lisa_cluster", "grocery_score")], strict=True
):
    p = bg[bg[col].notna()]
    p.plot(ax=ax, color=p[col].map(CL), edgecolor="white", linewidth=0.6)
    bg[bg[col].isna()].plot(ax=ax, color="none", edgecolor="#999", hatch="//", linewidth=0.4)
    ax.set_title(f"LISA clusters: {title} (p < {meta['lisa_alpha']})")
    ax.set_axis_off()
axes[1].legend(
    handles=[Patch(color=v, label=k) for k, v in CL.items()], loc="lower right", frameon=False
)
plt.tight_layout()

In [ ]:
cen = bg.geometry.centroid.to_crs(4326)
bg["lon"], bg["lat"] = cen.x.round(3), cen.y.round(3)
cols = [
    "geoid",
    "lisa_cluster",
    "lisa_cluster_fdr",
    "lisa_p",
    "lat",
    "lon",
    "access_score",
    "grocery_score",
    "transit_score",
    "median_hh_income",
    "below_pov_pct",
    "pop_65plus_pct",
    "pop_total",
]
cl = bg[bg.lisa_cluster.isin(["LL", "HH"])].sort_values(["lisa_cluster", "lat"])
print(cl.groupby("lisa_cluster")["lat"].agg(["min", "max", "count"]))
cl[cols].round(2)

## Caveats

- **ACS margins of error.** Block-group CVs above 30 % are common (zero-car share in most block
  groups, the poverty share in about half). The Monte Carlo ranges show how robust each result is,
  but they don't remove the error.
- **Ecological inference.** Groups are counted per block group and assumed to live in its homes like
  everyone else. "Low-income residents have better access" means low-income *block groups* have better
  access; it does not observe individual households.
- **Small n.** 45 populated block groups: correlations and LISA clusters are indicative. Clusters are
  also reported after an FDR correction.
- **Edge.** Six Douglas County block groups (centroid inside the bbox) are included; services up to
  2 km outside the bbox count.
- **Vintage.** ACS 5-year 2020–2024 against an Overture release from October 2026: places and
  roads are newer than the population figures.